In [9]:
# 1. Dataset Loading & Initial Data Quality Report
# Objective: Load raw data, check shape, duplicates, nulls, and types

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('fifa21_raw_data.csv', low_memory=False)

print("=== 1. DATA QUALITY REPORT ===")
print(f"Dataset Shape (Rows, Columns): {df.shape}")
print(f"Duplicate Rows Count: {df.duplicated().sum()}")

print("\n--- Missing Values Count per Column ---")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0])

print("\n--- Inconsistent Data Types Sample ---")
print(df[['ID', 'Joined', 'Height', 'Weight', 'Value', 'Wage', 'Release Clause']].dtypes)

print("\n--- Value Formatting Anomalies Sample ---")
print(df[['Height', 'Weight', 'Value', 'Wage', 'Hits']].head(3))


=== 1. DATA QUALITY REPORT ===
Dataset Shape (Rows, Columns): (18979, 77)
Duplicate Rows Count: 1

--- Missing Values Count per Column ---
Loan Date End    17966
dtype: int64

--- Inconsistent Data Types Sample ---
ID                int64
Joined              str
Height              str
Weight              str
Value               str
Wage                str
Release Clause      str
dtype: object

--- Value Formatting Anomalies Sample ---
  Height  Weight   Value   Wage   Hits
0   5'7"  159lbs  €67.5M  €560K  \n372
1   6'2"  183lbs    €46M  €220K  \n344
2   6'2"  192lbs    €75M  €125K   \n86


In [10]:
# 2. Duplicate Removal
# Objective: Identify and remove duplicate records to avoid skewed data

initial_rows = len(df)
duplicate_count = df.duplicated().sum()

# Drop exact duplicates
df = df.drop_duplicates().reset_index(drop=True)
removed_rows = initial_rows - len(df)

print("=== 2. DUPLICATE REMOVAL REPORT ===")
print(f"Duplicates identified: {duplicate_count}")
print(f"Duplicate rows removed: {removed_rows}")
print(f"Current Row Count: {len(df)}")


=== 2. DUPLICATE REMOVAL REPORT ===
Duplicates identified: 1
Duplicate rows removed: 1
Current Row Count: 18978


In [11]:
# 3. Missing Data Handling
# Justification:
# - 'Loan Date End': Missing values indicate permanent contract players (not on loan). Imputed with 'Not on Loan'.
# - 'Hits': Missing entries imputed with '0'.

df['Loan Date End'] = df['Loan Date End'].fillna('Not on Loan')
df['Hits'] = df['Hits'].fillna('0')

print("=== 3. MISSING DATA HANDLING ===")
print("Remaining Nulls in 'Loan Date End':", df['Loan Date End'].isnull().sum())
print("Remaining Nulls in 'Hits':", df['Hits'].isnull().sum())


=== 3. MISSING DATA HANDLING ===
Remaining Nulls in 'Loan Date End': 0
Remaining Nulls in 'Hits': 0


In [12]:
# 4. Standardisation & Unit Normalisation
# - Clean whitespace & hidden newline (\n) characters
# - Convert 'Hits' 'K' notations to numeric float
# - Standardize 'Height' to cm and 'Weight' to kg

# Clean text
df['Team & Contract'] = df['Team & Contract'].astype(str).str.strip().str.replace('\n', ' ')
df['Hits'] = df['Hits'].astype(str).str.replace('\n', '').str.strip()

# Standardize Hits
df['Hits'] = df['Hits'].apply(lambda x: float(x.replace('K', '')) * 1000 if 'K' in str(x) else float(x) if str(x).replace('.', '', 1).isdigit() else 0.0)

# Convert Height to cm
def convert_height(val):
    val_str = str(val).strip()
    if "'" in val_str:
        feet, inches = val_str.replace('"', '').split("'")
        return round(int(feet) * 30.48 + int(inches) * 2.54, 1)
    elif 'cm' in val_str:
        return float(val_str.replace('cm', ''))
    return float(val_str)

df['Height_cm'] = df['Height'].apply(convert_height)

# Convert Weight to kg
def convert_weight(val):
    val_str = str(val).strip()
    if 'lbs' in val_str:
        return round(float(val_str.replace('lbs', '')) * 0.45359237, 1)
    elif 'kg' in val_str:
        return float(val_str.replace('kg', ''))
    return float(val_str)

df['Weight_kg'] = df['Weight'].apply(convert_weight)

print("=== 4. STANDARDIZATION COMPLETE ===")
print(df[['Height', 'Height_cm', 'Weight', 'Weight_kg', 'Hits']].head(3))

=== 4. STANDARDIZATION COMPLETE ===
  Height  Height_cm  Weight  Weight_kg   Hits
0   5'7"      170.2  159lbs       72.1  372.0
1   6'2"      188.0  183lbs       83.0  344.0
2   6'2"      188.0  192lbs       87.1   86.0


In [13]:
# 5. Data Type Corrections & Currency Conversions
# - 'Joined' to datetime
# - 'ID' to string
# - Parse 'Value', 'Wage', 'Release Clause' (€, M, K to numeric float)

# Date and ID conversions
df['Joined'] = pd.to_datetime(df['Joined'], errors='coerce')
df['ID'] = df['ID'].astype(str)

# Currency parser function
def parse_currency(val):
    if isinstance(val, str):
        val = val.replace('€', '').strip()
        if 'M' in val:
            return float(val.replace('M', '')) * 1_000_000
        elif 'K' in val:
            return float(val.replace('K', '')) * 1_000
        try:
            return float(val)
        except:
            return 0.0
    return float(val)

df['Value_EUR'] = df['Value'].apply(parse_currency)
df['Wage_EUR'] = df['Wage'].apply(parse_currency)
df['Release_Clause_EUR'] = df['Release Clause'].apply(parse_currency)

print("=== 5. DATA TYPE CORRECTIONS COMPLETE ===")
print(df[['ID', 'Joined', 'Value_EUR', 'Wage_EUR', 'Release_Clause_EUR']].dtypes)


=== 5. DATA TYPE CORRECTIONS COMPLETE ===
ID                               str
Joined                datetime64[us]
Value_EUR                    float64
Wage_EUR                     float64
Release_Clause_EUR           float64
dtype: object


In [14]:
# 6. Outlier Detection (IQR Method) & Strategy
# Justification: Retain outliers as superstar footballers naturally have high market wages.

Q1 = df['Wage_EUR'].quantile(0.25)
Q3 = df['Wage_EUR'].quantile(0.75)
IQR = Q3 - Q1

lower_limit = max(0, Q1 - 1.5 * IQR)
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df['Wage_EUR'] < lower_limit) | (df['Wage_EUR'] > upper_limit)]

print("=== 6. OUTLIER DETECTION (IQR METHOD) ===")
print(f"Wage_EUR Q1: {Q1} EUR | Q3: {Q3} EUR | IQR: {IQR} EUR")
print(f"Upper Outlier Cutoff: {upper_limit} EUR")
print(f"Total Outliers Identified: {len(outliers)}")
print("Decision: Retained all outliers (real-world market wage variance).")

=== 6. OUTLIER DETECTION (IQR METHOD) ===
Wage_EUR Q1: 1000.0 EUR | Q3: 8000.0 EUR | IQR: 7000.0 EUR
Upper Outlier Cutoff: 18500.0 EUR
Total Outliers Identified: 2277
Decision: Retained all outliers (real-world market wage variance).


In [15]:
# 7. Before vs. After Summary Table & Export Clean Dataset

summary_df = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Duplicate Rows",
        "Loan Date End Nulls",
        "Height Data Type",
        "Weight Data Type",
        "Joined Column Type",
        "ID Column Type",
        "Value / Wage Type"
    ],
    "Before Cleaning": [
        "18,979",
        "1",
        "17,966",
        "object (e.g. 5'7\")",
        "object (e.g. 159lbs)",
        "object (string)",
        "int64",
        "object (e.g. €67.5M)"
    ],
    "After Cleaning": [
        f"{len(df):,}",
        "0",
        "0 (Imputed 'Not on Loan')",
        f"{df['Height_cm'].dtype} (cm in float)",
        f"{df['Weight_kg'].dtype} (kg in float)",
        f"{df['Joined'].dtype}",
        f"{df['ID'].dtype} (string)",
        f"{df['Value_EUR'].dtype} (EUR in float)"
    ]
})

print("=== 7. BEFORE VS. AFTER SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Export clean dataset to CSV
output_filename = "fifa21_cleaned_data.csv"
df.to_csv(output_filename, index=False)
print(f"\n[SUCCESS] Cleaned dataset saved to '{output_filename}' successfully.")

=== 7. BEFORE VS. AFTER SUMMARY TABLE ===
             Metric      Before Cleaning            After Cleaning
         Total Rows               18,979                    18,978
     Duplicate Rows                    1                         0
Loan Date End Nulls               17,966 0 (Imputed 'Not on Loan')
   Height Data Type   object (e.g. 5'7")     float64 (cm in float)
   Weight Data Type object (e.g. 159lbs)     float64 (kg in float)
 Joined Column Type      object (string)            datetime64[us]
     ID Column Type                int64              str (string)
  Value / Wage Type object (e.g. €67.5M)    float64 (EUR in float)

[SUCCESS] Cleaned dataset saved to 'fifa21_cleaned_data.csv' successfully.
